<a href="https://anirbansen2709.medium.com/finetuning-llms-using-lora-77fb02cbbc48">LLM Finetuning with LoRA</a>

In [1]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 4.5 MB/s eta 0:00:00


In [2]:
import torch
from torch.utils.data import Dataset
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import Markdown

In [3]:
from typing import Dict, List
from datasets import Dataset, load_dataset, disable_caching
disable_caching()

In [4]:
from functools import partial
import copy
from transformers import DataCollatorForSeq2Seq

In [5]:
# Load Dolly models

# device_map="auto" - To automatically distribute the model’s layers across
# the available hardware (CPU, GPU, multiple GPUs
instruction_pipeline = pipeline(
    model="databricks/dolly-v2-3b",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

instruct_pipeline.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/databricks/dolly-v2-3b:
- instruct_pipeline.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

Device set to use cpu


In [6]:
prompt = "What are the pricing options of Medium blogging website?"

In [7]:
generated_output = instruction_pipeline(prompt)

In [8]:
Markdown(generated_output[0]["generated_text"])

Medium allows users to subscribe at different monthly levels (Bronze, Silver, Gold, and Platinum). Each subscription level comes with different storage space, website statistics, and advertisement space. All of the pricing options are disclosed in detail on Medium's website.

In [9]:
# Dataset Preparation
dataset = load_dataset("MBZUAI/LaMini-instruction" , split = 'train')
small_dataset = dataset.select([i for i in range(200)])

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/871 [00:00<?, ?B/s]

data/train-00000-of-00003-929c6c373c0473(…):   0%|          | 0.00/203M [00:00<?, ?B/s]

data/train-00001-of-00003-1f823c156c353d(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00003-752db5e1576f50(…):   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2585615 [00:00<?, ? examples/s]

In [10]:
print(small_dataset[0])

{'instruction': 'List 5 reasons why someone should learn to code', 'response': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'instruction_source': 'alpaca'}


In [11]:
# Create templates
prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request. Instruction: {instruction}\n Response:"""
answer_template = """{response}"""

In [12]:
def _add_text(record):

  instruction = record["instruction"]
  response = record["response"]

  if not instruction:
        raise ValueError(f"Expected an instruction in: {record}")
  if not response:
        raise ValueError(f"Expected a response in: {record}")

  record["prompt"] = prompt_template.format(instruction=instruction)
  record["answer"] = answer_template.format(response=response)

  record["text"] = record["prompt"] + record["answer"]
  return record

In [13]:
# Iterate through the dataset and apply _add_text
small_dataset = small_dataset.map(_add_text)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [14]:
print(small_dataset[0])

{'instruction': 'List 5 reasons why someone should learn to code', 'response': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'instruction_source': 'alpaca', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request. Instruction: List 5 reasons why someone should learn to code\n Response:', 'answer': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request. Instruction: List 5 reasons why someone should learn to code\n Response:

In [15]:
# Load Tokenizer
tokenizer_model = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
# Whenever padding is required, just use the end-of-sequence token instead
tokenizer.pad_token = tokenizer.eos_token

In [16]:
tokenizer("Explain AI in one line")

{'input_ids': [1672, 19104, 14980, 275, 581, 1386], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [17]:
# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# Learn more about bit quantization

In [18]:
# Load model using AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    "databricks/dolly-v2-3b",
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

In [19]:
MAX_LENGTH = 256

In [20]:
# Generate tokens for batch processing
def _preprocess_batch(batch: Dict[str, List]):

  model_inputs = tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True, padding='max_length')
  # For casual language models, the inputs and outputs are same i.e, the model tries to predict the next token of the input sequence
  model_inputs["labels"] = copy.deepcopy(model_inputs['input_ids'])
  return model_inputs

In [21]:
_preprocessing_function = partial(_preprocess_batch)

In [ ]:
encoded_small_dataset = small_dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "response", "prompt", "answer"],
)

### Low-Rank Adaptation (LoRA)

**Key Idea:**  
Instead of updating the **entire weight matrix** during fine-tuning, LoRA introduces **two smaller matrices** that track the changes. These matrices are multiplied together to produce a matrix of the **same size** as the model’s original weight matrix.

---

#### Formula

**Fine-tuned Weights = Original Weights + LoRA Weight Changes**

---

#### How It Works

- LoRA performs a **matrix decomposition**, where the weight update is represented as the product of two smaller matrices:  
  **ΔW = A × B**
- This approach allows efficient adaptation with **fewer trainable parameters**.

---

#### Rank and Precision

- The **rank** of the smaller matrices determines the **precision** and **capacity** of the LoRA adaptation.  
- A **higher rank** allows more precise weight adjustments, while a **lower rank** reduces computation and memory usage.

---

**Reference:**  
<a href="https://youtu.be/t1caDsMzWBk?si=fsMvXpu27mfwE1I9" target="_blank">LoRA Explained (YouTube)</a>
